# IPL Teams Dataset — Detailed EDA and Data Cleaning

## Objective

This notebook examines the team reference table before it is used in the IPL analytics model. The goal is not to change names simply because they look unusual; it is to establish whether the table is complete, whether `team_id` can act as a key, and which naming or cross-table issues need a deliberate modelling decision.

The analysis follows a practical workflow: **inspect → ask a question → test it → record the decision → clean conservatively → validate → export**.

**Dataset grain:** one row is intended to represent one team identifier and its canonical team name.

## 1. Imports and data loading

First, load the file and preserve an untouched copy. Keeping `df_raw` lets us compare the source with the cleaned result and makes every cleaning change auditable.

In [16]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_rows", 60)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 100)

DATA_PATH = Path("../../teams_data.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"Source CSV not found: {DATA_PATH.resolve()}")

df_raw = pd.read_csv(DATA_PATH)
df = df_raw.copy()

print(f"Source file: {DATA_PATH.resolve()}")
print(f"Loaded shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Source file: C:\Users\Bhuvan\Desktop\IPL DATA\teams_data.csv
Loaded shape: 16 rows × 2 columns
Columns: ['team_id', 'team_name']


## 2. First look: what does one row represent?

Before transforming anything, inspect the rows and schema. A small reference table should be easy to reason about, but it still needs key and relationship checks.

In [17]:
display(df.head(10))

,team_id,team_name
0,1,Royal Challengers Bangalore
1,2,Sunrisers Hyderabad
2,3,Mumbai Indians
3,4,Rising Pune Supergiant
4,5,Gujarat Lions
5,6,Kolkata Knight Riders
6,129,Chennai Super Kings
7,134,Rajasthan Royals
8,252,Delhi Capitals
9,494,Punjab Kings


In [18]:
display(df.sample(min(5, len(df)), random_state=42))

,team_id,team_name
0,1,Royal Challengers Bangalore
1,2,Sunrisers Hyderabad
5,6,Kolkata Knight Riders
14,1419,Pune Warriors
13,1414,Kochi Tuskers Kerala


In [19]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null_count": df.notna().sum(),
    "null_count": df.isna().sum(),
    "distinct_values": df.nunique(dropna=True)
})
display(schema)

,dtype,non_null_count,null_count,distinct_values
team_id,int64,16,0,16
team_name,str,16,0,16


### Initial questions

- Does every record have a team ID and a team name?
- Is each team ID unique, as expected for a primary key?
- Are any complete rows repeated?
- Are there hidden blank strings or padded whitespace that ordinary null checks would miss?
- Do name variants represent legitimate historical names, or are they data issues that should be fixed only after checking context?

## 3. Completeness and duplicate checks

A null check alone is not enough for text columns: an empty string or a string made only of spaces is technically non-null. Check both kinds before deciding whether anything needs cleaning.

In [20]:
null_summary = pd.DataFrame({
    "null_count": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "blank_or_whitespace_only": [
        int(df[c].astype("string").str.strip().eq("").fillna(False).sum())
        if pd.api.types.is_object_dtype(df[c]) or pd.api.types.is_string_dtype(df[c])
        else 0
        for c in df.columns
    ]
})
display(null_summary)
print(f"Exact duplicate rows: {df.duplicated().sum()}")
print(f"Rows duplicated on team_id: {df['team_id'].duplicated().sum()}")
print(f"Rows duplicated on team_name: {df['team_name'].duplicated().sum()}")

,null_count,null_pct,blank_or_whitespace_only
team_id,0,0.0,0
team_name,0,0.0,0


Exact duplicate rows: 0
Rows duplicated on team_id: 0
Rows duplicated on team_name: 0


**Decision checkpoint:** if the table has no nulls, blanks, or duplicates, there is no justification for deleting records. Continue with key and naming checks rather than applying generic cleaning steps.

## 4. Investigate `team_id` — candidate primary key

The IDs are not expected to be sequential unless the source specification says so. The useful checks are whether the field is missing, duplicated, numeric, and stable enough to identify each row—not whether it contains every integer in a range.

In [21]:
print("team_id dtype:", df["team_id"].dtype)
print("Missing team_id values:", int(df["team_id"].isna().sum()))
print("Unique team_id values:", int(df["team_id"].nunique(dropna=True)))
print("team_id is unique:", bool(df["team_id"].is_unique))
print("team_id values (sorted):", sorted(df["team_id"].dropna().tolist()))

team_id dtype: int64
Missing team_id values: 0
Unique team_id values: 16
team_id is unique: True
team_id values (sorted): [1, 2, 3, 4, 5, 6, 129, 134, 252, 494, 614, 615, 1068, 1414, 1419, 3604]


In [22]:
sorted_ids = sorted(df["team_id"].dropna().tolist())
gaps = [(left, right, right - left) for left, right in zip(sorted_ids, sorted_ids[1:]) if right - left > 1]
print(f"IDs form a consecutive sequence: {sorted_ids == list(range(sorted_ids[0], sorted_ids[0] + len(sorted_ids))) if sorted_ids else True}")
print(f"Number of gaps between neighbouring IDs: {len(gaps)}")
print("A few gaps:", gaps[:10])

IDs form a consecutive sequence: False
Number of gaps between neighbouring IDs: 9
A few gaps: [(6, 129, 123), (129, 134, 5), (134, 252, 118), (252, 494, 242), (494, 614, 120), (615, 1068, 453), (1068, 1414, 346), (1414, 1419, 5), (1419, 3604, 2185)]


**Interpretation:** non-sequential IDs are not, by themselves, a defect in a reference table. Do not renumber them: other IPL datasets may use these values as foreign keys. If IDs are unique and complete, retain them unchanged.

## 5. Investigate `team_name`

Team names are human-readable labels, so whitespace and duplicates matter. However, different names can also be historically meaningful. Check the literal values before normalizing anything.

In [23]:
name_series = df["team_name"].astype("string")
padded_mask = name_series.ne(name_series.str.strip()).fillna(False)
blank_mask = name_series.str.strip().eq("").fillna(False)

print("Missing team_name values:", int(df["team_name"].isna().sum()))
print("Whitespace-padded names:", int(padded_mask.sum()))
print("Blank/whitespace-only names:", int(blank_mask.sum()))
print("Unique team names:", int(df["team_name"].nunique(dropna=True)))
print("team_name is unique:", bool(df["team_name"].is_unique))

Missing team_name values: 0
Whitespace-padded names: 0
Blank/whitespace-only names: 0
Unique team names: 16
team_name is unique: True


In [24]:
display(df[["team_id", "team_name"]].sort_values("team_id").reset_index(drop=True))

,team_id,team_name
0,1,Royal Challengers Bangalore
1,2,Sunrisers Hyderabad
2,3,Mumbai Indians
3,4,Rising Pune Supergiant
4,5,Gujarat Lions
5,6,Kolkata Knight Riders
6,129,Chennai Super Kings
7,134,Rajasthan Royals
8,252,Delhi Capitals
9,494,Punjab Kings


### 5.1 Historical name variants: Rising Pune

Two rows use `Rising Pune Supergiant` and `Rising Pune Supergiants`. The IDs differ as well. Treat this as a potential historical/team-identity modelling question, not an automatic duplicate: keep the source records intact unless the project defines a canonical franchise entity that intentionally merges them.

In [25]:
rising_pune = df[df["team_name"].astype("string").str.contains("Rising Pune", case=False, na=False)]
display(rising_pune.sort_values("team_id"))
print("Distinct IDs in these rows:", rising_pune["team_id"].nunique())

,team_id,team_name
3,4,Rising Pune Supergiant
15,3604,Rising Pune Supergiants


Distinct IDs in these rows: 2


### 5.2 Historical entity mismatch: Deccan Chargers

The team table contains `Deccan Chargers` with `team_id = 1068`. In the uploaded team-aliases table, the `Deccan Chargers` alias points to `team_id = 2`, whose canonical name here is `Sunrisers Hyderabad`. This is a **cross-table relationship conflict**. Do not silently rewrite either ID in this EDA; record it for resolution against the source system/business rules before joins or dashboard measures depend on it.

In [26]:
deccan_rows = df[df["team_name"].astype("string").str.contains("Deccan Chargers", case=False, na=False)]
display(deccan_rows)

# Optional cross-reference: run only when the alias file is available.
alias_candidates = [Path("team_aliases(1).csv"), Path("/mnt/data/team_aliases(1).csv"), Path("team_aliases.csv")]
ALIAS_PATH = next((p for p in alias_candidates if p.exists()), None)
if ALIAS_PATH is not None:
    aliases = pd.read_csv(ALIAS_PATH)
    deccan_alias = aliases[aliases["alias_name"].astype("string").str.casefold().eq("deccan chargers")]
    print(f"Alias source: {ALIAS_PATH}")
    display(deccan_alias)
    print("Canonical team record for the alias team_id:")
    display(df[df["team_id"].isin(deccan_alias["team_id"])][["team_id", "team_name"]])
else:
    print("Alias file not found in expected locations; cross-table check skipped.")

,team_id,team_name
12,1068,Deccan Chargers


Alias file not found in expected locations; cross-table check skipped.


## 6. Conservative cleaning

The source checks above determine the cleaning actions. For this file, only trim surrounding whitespace in `team_name` if any is present. Do not drop rows, renumber IDs, merge similar names, or impute values without a source-backed reason.

In [27]:
df_clean = df.copy()

# Normalize surrounding whitespace only; this is a formatting correction, not a name remapping.
df_clean["team_name"] = df_clean["team_name"].astype("string").str.strip()

# Preserve team_id values as supplied; no sequential renumbering or inferred ID correction.
print("Whitespace corrections applied:", int(padded_mask.sum()))
print("Rows removed:", len(df) - len(df_clean))
print("Source team IDs changed:", not df_clean["team_id"].equals(df["team_id"]))

Whitespace corrections applied: 0
Rows removed: 0
Source team IDs changed: False


## 7. Final validation

Re-run the important checks on the cleaned dataframe. Export only after these assertions pass; this prevents a cleaned file from being written if key assumptions have been broken.

In [28]:
validation = {
    "same_row_count": len(df_clean) == len(df_raw),
    "same_column_order": df_clean.columns.tolist() == df_raw.columns.tolist(),
    "no_null_team_id": not df_clean["team_id"].isna().any(),
    "team_id_unique": df_clean["team_id"].is_unique,
    "no_null_team_name": not df_clean["team_name"].isna().any(),
    "no_blank_team_name": not df_clean["team_name"].astype("string").str.strip().eq("").fillna(False).any(),
    "no_exact_duplicate_rows": not df_clean.duplicated().any(),
    "team_names_unique": df_clean["team_name"].is_unique,
}
display(pd.DataFrame({"check": validation.keys(), "passed": validation.values()}))
assert all(validation.values()), "One or more final validation checks failed."
print("All final validation checks passed.")

,check,passed
0,same_row_count,True
1,same_column_order,True
2,no_null_team_id,True
3,team_id_unique,True
4,no_null_team_name,True
5,no_blank_team_name,True
6,no_exact_duplicate_rows,True
7,team_names_unique,True


All final validation checks passed.


## 8. Export cleaned dataset

Write the cleaned reference table without a pandas index column. The file retains the original row count, column order, and team IDs.

In [29]:
from pathlib import Path

OUTPUT_DIR = Path("../../data/cleaned")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OUTPUT_PATH = OUTPUT_DIR / "teams_data_clean.csv"
df_clean.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

df_export_check = pd.read_csv(OUTPUT_PATH)

print(f"Exported to: {OUTPUT_PATH.resolve()}")
print(f"Exported shape: {df_export_check.shape}")
print(f"Exported columns: {df_export_check.columns.tolist()}")

assert df_export_check.shape == df_clean.shape
assert df_export_check.columns.tolist() == df_clean.columns.tolist()
assert df_export_check["team_id"].is_unique

print("Export reload check passed.")

Exported to: C:\Users\Bhuvan\Desktop\IPL DATA\data\cleaned\teams_data_clean.csv
Exported shape: (16, 2)
Exported columns: ['team_id', 'team_name']
Export reload check passed.


## 9. Findings and decisions

| Area | Finding | Decision |
|---|---|---|
| Dataset grain | One row per listed team ID | Preserve each source record |
| Completeness | No missing IDs or names in the inspected source | No imputation required |
| Duplicates | No exact duplicate rows; IDs and names are unique | No deduplication required |
| `team_id` | IDs are non-sequential | Preserve IDs; do not renumber |
| Team names | Similar/historical names appear in the table | Keep source names unless a documented canonicalization rule says otherwise |
| Rising Pune | Singular/plural name variants have different IDs | Retain both; validate intended modelling grain |
| Deccan Chargers | Alias file maps the name to ID 2 while the teams table lists ID 1068 | Resolve this cross-table conflict before joins |
| Cleaning | Only surrounding whitespace is trimmed if needed | Conservative, auditable transformation |
| Export | Cleaned table written without index | Ready as a reference input, subject to resolving the noted relationship conflict |

**Overall assessment:** the table is structurally clean for a small reference dataset. The main risk is not missing data; it is consistent entity mapping across datasets. Resolve the Deccan Chargers ID mismatch before relying on `team_id` relationships in a unified IPL model.

## 10. Reproducibility note

This notebook deliberately separates data cleaning from business-rule reconciliation. If the canonical team identity rules change, update the mapping explicitly, document the source of the decision, and add a validation test rather than editing names or IDs silently.